<a href="https://colab.research.google.com/github/IrynaSukhoverkhova/my-first-work-place/blob/main/HW_16_2_%D0%9F%D1%80%D0%BE%D0%B3%D0%BD%D0%BE%D0%B7%D1%83%D0%B2%D0%B0%D0%BD%D0%BD%D1%8F_%D0%BA%D1%80%D0%B5%D0%B4%D0%B8%D1%82%D0%BD%D0%BE%D0%B3%D0%BE_%D1%80%D0%B8%D0%B7%D0%B8%D0%BA%D1%83.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнє завдання: Прогнозування кредитного ризику

## Мета завдання
Застосувати знання з лекції для побудови моделі логістичної регресії, що прогнозує ймовірність дефолту за кредитом, іншими словами, що людина не поверне кредит. Ви пройдете весь цикл: від дослідницького аналізу до оцінки якості класифікаційної моделі.

## Опис датасету
**Credit Risk Dataset** містить інформацію про 32,000+ позичальників з такими параметрами:
- **person_age**: Вік позичальника
- **person_income**: Річний дохід
- **person_home_ownership**: Тип володіння житлом
- **person_emp_length**: Стаж роботи (в роках)
- **loan_intent**: Мета кредиту
- **loan_grade**: Кредитний рейтинг (A - кращий, F - гірший)
- **loan_amnt**: Сума кредиту
- **loan_int_rate**: Процентна ставка
- **loan_status**: Статус кредиту (0 = сплачено, 1 = дефолт) - **цільова змінна**
- **loan_percent_income**: Відношення кредиту до доходу
- **cb_person_default_on_file**: Історія дефолтів (Y/N)
- **cb_person_cred_hist_length**: Довжина кредитної історії

---

## Завдання 1: Завантаження та перший огляд даних (1 бал)

**Що потрібно зробити:**
1. Завантажте дані з файлу `credit_risk_dataset.csv`
2. Виведіть розмір датасету
3. Покажіть перші 5 рядків
4. Виведіть загальну інформацію про дані (кількість записів, типи колонок)
5. Перевірте розподіл цільової змінної (відсотк даних для кожного класу)

Дайте висновок, це задача збалансованої чи незбалансованої класифікації.


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Завантаження даних
df = pd.read_csv('credit_risk_dataset.csv')

In [2]:
# 2. Розмір датасету
print(f"Розмір датасету: {df.shape[0]} рядків, {df.shape[1]} колонок")

Розмір датасету: 32581 рядків, 12 колонок


In [3]:
# 3. Перші 5 рядків
display(df.head())

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [4]:
# 4. Загальна інформація про дані
print("\nЗагальна інформація:")
df.info()


Загальна інформація:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  object 
 3   person_emp_length           31686 non-null  float64
 4   loan_intent                 32581 non-null  object 
 5   loan_grade                  32581 non-null  object 
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               29465 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  object 
 11  cb_person_cred_hist_length  32581 non-null  int64  
dtypes: float64(3), int64(5), object(4)
memory usage: 3.0+ MB


In [5]:
# 5. Розподіл цільової змінної
target_counts = df['loan_status'].value_counts().sort_index()
target_pct = df['loan_status'].value_counts(normalize=True).sort_index() * 100

target_df = pd.DataFrame({
    'Клас': ['0 - сплачено', '1 - дефолт'],
    'Кількість': target_counts.values,
    'Відсоток (%)': target_pct.round(2).values
})
display(target_df)

# Візуалізація розподілу
fig = px.bar(
    target_df,
    x='Клас',
    y='Кількість',
    text=target_df['Відсоток (%)'].astype(str) + '%',
    color='Клас',
    color_discrete_map={
        '0 - сплачено': '#264653',  # темно-бірюзовий
        '1 - дефолт': '#F4A261'     # пісочний
    },
    title='Розподіл цільової змінної loan_status'
)
fig.update_traces(textposition='outside')
fig.show()

# Перевірка збалансованості
minority_pct = target_pct.min()
if minority_pct < 40:
    print(f"\nВисновок: менший клас становить лише {minority_pct:.2f}% даних — "
          f"це задача НЕЗБАЛАНСОВАНОЇ класифікації.")
else:
    print(f"\nВисновок: класи розподілені приблизно рівно — задача збалансованої класифікації.")

,Клас,Кількість,Відсоток (%)
0,0 - сплачено,25473,78.18
1,1 - дефолт,7108,21.82



Висновок: менший клас становить лише 21.82% даних — це задача НЕЗБАЛАНСОВАНОЇ класифікації.



---

## Завдання 2: Дослідницький аналіз даних (EDA) (4 бали)

**Що потрібно зробити:**
1. Обчисліть відсоток пропущених значень в колонках. За наявності пропущених значень - заповніть їх медіаною для числових колонок і найбільш частим значеннмя для категоріальних.
2. Проаналізуйте розподіл числових змінних.
3. Знайдіть та обробіть викиди в колонці person_income з допомогою [Interquartile range](https://uk.wikipedia.org/wiki/%D0%9C%D1%96%D0%B6%D0%BA%D0%B2%D0%B0%D1%80%D1%82%D0%B8%D0%BB%D1%8C%D0%BD%D0%B8%D0%B9_%D1%80%D0%BE%D0%B7%D0%BC%D0%B0%D1%85).
4. Проаналізуйте категоріальні змінні відносно цільової та частоту зустрічання різних значень в них.
5. Візуалізуйте взаємозв'язок ознак з цільовою змінною.


In [6]:
# 1. Пропущені значення

# Відсоток пропусків у кожній колонці
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = missing_pct[missing_pct > 0].sort_values(ascending=False).to_frame('Пропуски (%)')
print("Колонки з пропущеними значеннями:")
display(missing_df)

# Розділення колонок на числові та категоріальні
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.drop('loan_status')
cat_cols = df.select_dtypes(include=['object']).columns

# Заповнення пропусків
for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print(f"\nПропусків після заповнення: {df.isnull().sum().sum()}")

Колонки з пропущеними значеннями:


,Пропуски (%)
loan_int_rate,9.56
person_emp_length,2.75



Пропусків після заповнення: 0


In [7]:
#2. Розподіл числових змінних

display(df[num_cols].describe().round(2))

fig = make_subplots(rows=3, cols=3, subplot_titles=list(num_cols))
for i, col in enumerate(num_cols):
    row, c = i // 3 + 1, i % 3 + 1
    fig.add_trace(go.Histogram(x=df[col], marker_color='#264653', name=col, nbinsx=50), row=row, col=c)

fig.update_layout(height=800, title_text='Розподіл числових змінних', showlegend=False)
fig.show()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length
count,32581.00,32581.00,32581.00,32581.00,32581.00,32581.00,32581.00
mean,27.73,66074.85,4.77,9589.37,11.01,0.17,5.80
std,6.35,61983.12,4.09,6322.09,3.08,0.11,4.06
min,20.00,4000.00,0.00,500.00,5.42,0.00,2.00
25%,23.00,38500.00,2.00,5000.00,8.49,0.09,3.00
50%,26.00,55000.00,4.00,8000.00,10.99,0.15,4.00
75%,30.00,79200.00,7.00,12200.00,13.11,0.23,8.00
max,144.00,6000000.00,123.00,35000.00,23.22,0.83,30.00


**Аналіз розподілу числових змінних**

**person_age (вік)**. Розподіл має сильну правосторонню асиметрію. Більшість позичальників молоді: медіана становить 26 років, а 50% клієнтів мають від 23 до 30 років. Максимум у 144 роки нереальний, тобто це помилка в даних. Значення понад 100 років варто видалити або обробити як аномалії.

**person_income (річний дохід).** Розподіл також сильно скошений вправо. Медіана дорівнює 55 000, 75% позичальників заробляють до 79 200, а максимум сягає 6 000 000. На гістограмі майже всі значення зібрані в першому стовпчику, бо поодинокі дуже високі доходи розтягують шкалу.

**person_emp_length (стаж роботи).** Більшість має стаж від 2 до 7 років, медіана становить 4 роки. Максимум у 123 роки неможливий, отже, це помилка в даних. Розподіл скошений вправо.

**loan_amnt (сума кредиту).** Значення лежать у межах від 500 до 35 000, медіана дорівнює 8 000. Розподіл помірно скошений вправо. Помітні піки на круглих сумах (5 000, 10 000, 15 000, 20 000, 25 000), бо люди зазвичай просять круглі суми.

**loan_int_rate (процентна ставка).** Ставки коливаються від 5,42% до 23,22%, медіана становить 10,99%. Різкий пік близько 11% — це штучний ефект від заповнення пропусків медіаною: приблизно 10% значень замінили одним числом. Розподіл має кілька піків, що, ймовірно, відповідає різним кредитним рейтингам.

**loan_percent_income (кредит / дохід).** Розподіл скошений вправо. Для більшості позичальників кредит становить 9–23% річного доходу, медіана дорівнює 15%. Максимальне значення 0,83 означає, що кредит сягає 83% доходу - це потенційно ризикові випадки.

**cb_person_cred_hist_length (довжина кредитної історії).** Значення лежать у межах від 2 до 30 років, медіана становить 4 роки. Більшість клієнтів мають коротку кредитну історію, що узгоджується з їхнім молодим віком. Розподіл скошений вправо.

**Загальний висновок.** Майже всі числові змінні мають правосторонню асиметрію: більшість значень невеликі, і є довгий хвіст великих значень. У колонках person_age, person_emp_length та person_income є явні аномалії й викиди, які потрібно обробити перед побудовою моделі. Крім того, ознаки мають дуже різні масштаби (від десятих часток до мільйонів), тому для логістичної регресії їх треба буде масштабувати.

In [8]:
# 3. Викиди в person_income (IQR)

Q1 = df['person_income'].quantile(0.25)
Q3 = df['person_income'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['person_income'] < lower_bound) | (df['person_income'] > upper_bound)]
print(f"Q1 = {Q1:,.0f}, Q3 = {Q3:,.0f}, IQR = {IQR:,.0f}")
print(f"Межі: [{lower_bound:,.0f}; {upper_bound:,.0f}]")
print(f"Кількість викидів: {len(outliers)} ({len(outliers) / len(df) * 100:.2f}%)")

# Boxplot до обробки
fig = px.box(df, y='person_income', title='person_income до обробки викидів',
             color_discrete_sequence=['#264653'])
fig.show()

# Обробка: обрізаємо значення до меж IQR (capping)
df['person_income'] = df['person_income'].clip(lower=lower_bound, upper=upper_bound)

# Boxplot після обробки
fig = px.box(df, y='person_income', title='person_income після обробки викидів',
             color_discrete_sequence=['#F4A261'])
fig.show()

Q1 = 38,500, Q3 = 79,200, IQR = 40,700
Межі: [-22,550; 140,250]
Кількість викидів: 1484 (4.55%)


**До обробки.** Boxplot майже повністю сплющений біля нуля. Основна частина даних (від 38 500 до 79 200) візуально зливається в тонку лінію, бо шкалу розтягують екстремальні значення. Над верхнім вусом видно щільну смугу викидів до приблизно 1 млн, кілька точок на рівні 1,2–2 млн і одне екстремальне значення у 6 млн. Воно приблизно в 100 разів перевищує медіану і, швидше за все, є помилкою в даних.

**Після обробки.** Значення понад 140 250 замінено верхньою межею (capping). Тепер boxplot має нормальний вигляд: медіана близько 55 000, міжквартильний діапазон від 38 500 до 79 200, мінімум 4 000, максимум 140 250. Медіана лишилася незмінною, а розподіл став значно компактнішим і придатнішим для моделі.

**Чому обрала заміну, а не видалення.** Видалення прибрало б частину рядків, а отже, і корисну інформацію з інших колонок (суми кредиту, ставки, цільової змінної). Заміна межею зберігає всі записи і водночас прибирає вплив екстремальних значень.

**Висновок.** Метод IQR виявив викиди лише у верхній частині розподілу. Після їх обмеження значенням 140 250 змінна person_income має адекватний діапазон, а аномальні доходи (до 6 млн) більше не спотворюватимуть навчання моделі.

In [9]:
 # 4. Категоріальні змінні відносно цільової

for col in cat_cols:
    print(f"\n===== {col} =====")
    summary = df.groupby(col).agg(
        Кількість=('loan_status', 'count'),
        Частка_дефолтів_pct=('loan_status', 'mean')
    )
    summary['Частка (%)'] = (summary['Кількість'] / len(df) * 100).round(2)
    summary['Частка_дефолтів_pct'] = (summary['Частка_дефолтів_pct'] * 100).round(2)
    display(summary.sort_values('Частка_дефолтів_pct', ascending=False))

    # Частота значень з розбивкою за класом
    plot_df = df.groupby([col, 'loan_status']).size().reset_index(name='Кількість')
    plot_df['loan_status'] = plot_df['loan_status'].map({0: '0 - сплачено', 1: '1 - дефолт'})
    fig = px.bar(plot_df, x=col, y='Кількість', color='loan_status', barmode='group',
                 color_discrete_map={'0 - сплачено': '#264653', '1 - дефолт': '#F4A261'},
                 title=f'Розподіл {col} за loan_status')
    fig.show()


===== person_home_ownership =====


,Кількість,Частка_дефолтів_pct,Частка (%)
person_home_ownership,,,
RENT,16446,31.57,50.48
OTHER,107,30.84,0.33
MORTGAGE,13444,12.57,41.26
OWN,2584,7.47,7.93



===== loan_intent =====


,Кількість,Частка_дефолтів_pct,Частка (%)
loan_intent,,,
DEBTCONSOLIDATION,5212,28.59,16.00
MEDICAL,6071,26.70,18.63
HOMEIMPROVEMENT,3605,26.10,11.06
PERSONAL,5521,19.89,16.95
EDUCATION,6453,17.22,19.81
VENTURE,5719,14.81,17.55



===== loan_grade =====


,Кількість,Частка_дефолтів_pct,Частка (%)
loan_grade,,,
G,64,98.44,0.20
F,241,70.54,0.74
E,964,64.42,2.96
D,3626,59.05,11.13
C,6458,20.73,19.82
B,10451,16.28,32.08
A,10777,9.96,33.08



===== cb_person_default_on_file =====


,Кількість,Частка_дефолтів_pct,Частка (%)
cb_person_default_on_file,,,
Y,5745,37.81,17.63
N,26836,18.39,82.37


Зв'язок із дефолтом дуже сильний і чітко монотонний. Що гірший рейтинг, то вищий ризик:

**A, B, C:** переважна більшість кредитів сплачена, частка дефолтів невелика (приблизно 10–20%).

**D:** кількість дефолтів (помаранчевий стовпчик) уже перевищує кількість сплачених кредитів, тобто дефолтів понад 50%.

**E, F, G:** дефолти помітно переважають. Для G майже всі кредити закінчуються дефолтом.

**Висновок:** loan_grade — одна з найважливіших ознак для моделі. Рейтинг відображає оцінку ризику, яку банк уже зробив, і вона добре узгоджується з фактичними дефолтами. Ознака порядкова (A < B < … < G), тому її можна закодувати як числа 1–7 (Ordinal Encoding) і зберегти цей порядок.

**cb_person_default_on_file (історія дефолтів)**

**Частота:** 82,37% позичальників (26 836) раніше не мали дефолтів (**N**), а 17,63% (5 745) мали (**Y**).

**Зв'язок із дефолтом:**

**N:** частка дефолтів 18,39%
**Y:** частка дефолтів 37,81%

Позичальники з дефолтом в історії мають приблизно вдвічі вищий ризик не повернути новий кредит. Водночас більшість із них все ж сплачують кредит, тому сама собою ця ознака не визначає результат.

**Висновок:** **cb_person_default_on_file** — корисна бінарна ознака з помітним впливом на ризик, хоча й слабшим, ніж у **loan_grade**. Для моделі її варто закодувати як 0/1 (N = 0, Y = 1).

In [10]:
# 5. Взаємозв'язок ознак з цільовою змінною

df_plot = df.copy()
df_plot['loan_status'] = df_plot['loan_status'].map({0: '0 - сплачено', 1: '1 - дефолт'})

cols = list(num_cols.drop('loan_status')) if 'loan_status' in num_cols else list(num_cols)
colors = {'0 - сплачено': '#264653', '1 - дефолт': '#F4A261'}

n_cols = 2
n_rows = (len(cols) + n_cols - 1) // n_cols

fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=cols,
                    vertical_spacing=0.08, horizontal_spacing=0.1)

for i, col in enumerate(cols):
    row, c = i // n_cols + 1, i % n_cols + 1
    for status in ['0 - сплачено', '1 - дефолт']:
        fig.add_trace(
            go.Box(y=df_plot.loc[df_plot['loan_status'] == status, col],
                   name=status, marker_color=colors[status],
                   showlegend=(i == 0)),
            row=row, col=c
        )

fig.update_layout(height=350 * n_rows, title_text='Числові ознаки залежно від loan_status')
fig.show()

In [11]:
# Кореляція числових ознак з цільовою
cols_for_corr = [c for c in num_cols if c != 'loan_status'] + ['loan_status']
corr = df[cols_for_corr].corr()

fig = px.imshow(corr.round(2), text_auto=True, color_continuous_scale='RdBu_r',
                zmin=-1, zmax=1, title='Матриця кореляцій')
fig.update_layout(height=650)
fig.show()

print("Кореляція ознак з loan_status:")
print(corr['loan_status'].drop('loan_status').sort_values(ascending=False).round(3))

Кореляція ознак з loan_status:
loan_percent_income           0.379
loan_int_rate                 0.319
loan_amnt                     0.105
cb_person_cred_hist_length   -0.016
person_age                   -0.022
person_emp_length            -0.082
person_income                -0.246
Name: loan_status, dtype: float64


**Висновки: взаємозв'язок ознак з цільовою змінною**

**loan_percent_income (0,38)** має найсильніший зв'язок із дефолтом: чим більша частка доходу йде на кредит, тим вищий ризик.

**loan_int_rate (0,32)** має помітний позитивний зв'язок: вища ставка означає вищий ризик (ставка відображає кредитний рейтинг).

**person_income (−0,25)** має від'ємний зв'язок: що вищий дохід, то менша ймовірність дефолту.

**loan_amnt (0,11)** пов'язана з дефолтом слабко позитивно.

**person_emp_length (−0,08), person_age (−0,02) та cb_person_cred_hist_length (−0,02)** майже не впливають на дефолт. Boxplot кредитної історії це підтверджує: розподіли для обох класів практично однакові.

**Мультиколінеарність:** **person_age і cb_person_cred_hist_length** сильно корелюють між собою (0,86). Для моделі можна залишити лише одну з них.

Також корелюють **loan_amnt і loan_percent_income (0,57)**, а також person_income і loan_amnt (0,42). Це логічно, бо loan_percent_income розраховується з цих двох величин.

Серед категоріальних ознак **найсильніше впливають** loan_grade (від рейтингу D дефолтів більше, ніж сплачених кредитів) і cb_person_default_on_file (ризик удвічі вищий за наявності дефолтів у минулому).

**Ключові предиктори для моделі:** loan_percent_income, loan_int_rate, loan_grade, person_income, cb_person_default_on_file.


---

## Завдання 3: Аналіз кореляцій та Feature Engineering (3 бали)

**Що потрібно зробити:**
1. Побудуйте матрицю кореляцій для числових змінних.
2. Закодуйте категоріальні змінні.
3. Виберіть фінальний набір ознак, можна лишити всі, якщо ви вважаєте, що це - доцільно.



In [12]:
# 1. Матриця кореляцій для числових змінних

num_cols = df.select_dtypes(include=['int64', 'float64']).columns

corr = df[num_cols].corr()
fig = px.imshow(corr.round(2), text_auto=True, color_continuous_scale='RdBu_r',
                zmin=-1, zmax=1, title='Матриця кореляцій числових змінних')
fig.update_layout(height=650)
fig.show()

# Пари ознак із сильною кореляцією (|r| > 0.5)
print("Сильно корельовані пари ознак (|r| > 0.5):")
for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        r = corr.iloc[i, j]
        if abs(r) > 0.5:
            print(f"  {corr.columns[i]} — {corr.columns[j]}: {r:.2f}")

Сильно корельовані пари ознак (|r| > 0.5):
  person_age — cb_person_cred_hist_length: 0.86
  loan_amnt — loan_percent_income: 0.57


In [15]:
# 2. Кодування категоріальних змінних

df_encoded = df.copy()

# loan_grade — порядкова ознака: A (найкращий) → G (найгірший)
grade_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7}
df_encoded['loan_grade'] = df_encoded['loan_grade'].map(grade_map)

# cb_person_default_on_file — бінарна ознака
df_encoded['cb_person_default_on_file'] = df_encoded['cb_person_default_on_file'].map({'N': 0, 'Y': 1})

# person_home_ownership та loan_intent — номінальні ознаки → One-Hot Encoding
df_encoded = pd.get_dummies(df_encoded,
                            columns=['person_home_ownership', 'loan_intent'],
                            drop_first=True, dtype=int)

print(f"Розмір після кодування: {df_encoded.shape}")
display(df_encoded.head())
df_encoded.info()

Розмір після кодування: (32581, 18)


,person_age,person_income,person_emp_length,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
0,22,59000,123.0,4,35000,16.02,1,0.59,1,3,0,0,1,0,0,0,1,0
1,21,9600,5.0,2,1000,11.14,0,0.10,0,2,0,1,0,1,0,0,0,0
2,25,9600,1.0,3,5500,12.87,1,0.57,0,3,0,0,0,0,0,1,0,0
3,23,65500,4.0,3,35000,15.23,1,0.53,0,2,0,0,1,0,0,1,0,0
4,24,54400,8.0,3,35000,14.27,1,0.55,1,4,0,0,1,0,0,1,0,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   person_age                   32581 non-null  int64  
 1   person_income                32581 non-null  int64  
 2   person_emp_length            32581 non-null  float64
 3   loan_grade                   32581 non-null  int64  
 4   loan_amnt                    32581 non-null  int64  
 5   loan_int_rate                32581 non-null  float64
 6   loan_status                  32581 non-null  int64  
 7   loan_percent_income          32581 non-null  float64
 8   cb_person_default_on_file    32581 non-null  int64  
 9   cb_person_cred_hist_length   32581 non-null  int64  
 10  person_home_ownership_OTHER  32581 non-null  int64  
 11  person_home_ownership_OWN    32581 non-null  int64  
 12  person_home_ownership_RENT   32581 non-null  int64  
 13  loan_intent_EDUC

In [16]:
# 3. Фінальний набір ознак

# Видаляємо cb_person_cred_hist_length через сильну кореляцію з person_age (0.86)
# і майже нульовий зв'язок із цільовою змінною (-0.02)
features_to_drop = ['cb_person_cred_hist_length']

X = df_encoded.drop(columns=['loan_status'] + features_to_drop)
y = df_encoded['loan_status']

print(f"Кількість ознак у фінальному наборі: {X.shape[1]}")
print("Фінальні ознаки:")
for col in X.columns:
    print(f"  - {col}")

# Кореляція фінальних ознак з цільовою
final_corr = pd.concat([X, y], axis=1).corr()['loan_status'].drop('loan_status').sort_values()
fig = px.bar(x=final_corr.values, y=final_corr.index, orientation='h',
             color=final_corr.values, color_continuous_scale=['#264653', '#F4A261'],
             title='Кореляція фінальних ознак з loan_status',
             labels={'x': 'Кореляція', 'y': 'Ознака'})
fig.update_layout(height=600)
fig.show()


Кількість ознак у фінальному наборі: 16
Фінальні ознаки:
  - person_age
  - person_income
  - person_emp_length
  - loan_grade
  - loan_amnt
  - loan_int_rate
  - loan_percent_income
  - cb_person_default_on_file
  - person_home_ownership_OTHER
  - person_home_ownership_OWN
  - person_home_ownership_RENT
  - loan_intent_EDUCATION
  - loan_intent_HOMEIMPROVEMENT
  - loan_intent_MEDICAL
  - loan_intent_PERSONAL
  - loan_intent_VENTURE



---

## Завдання 4: Підготовка даних та навчання моделі (3 бали)

**Що потрібно зробити:**
1. Розділіть дані на X та y
2. Поділіть на навчальну та тестову вибірки
3. Застосуйте масштабування
4. Навчіть модель логістичної регресії
5. Зробіть прогнози на тренувальній та тестовій вибірках.



In [17]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# 1. Розділення на X та y
X = df_encoded.drop(columns=['loan_status', 'cb_person_cred_hist_length'])
y = df_encoded['loan_status']
print(f"X: {X.shape}, y: {y.shape}")

X: (32581, 16), y: (32581,)


In [18]:
# 2. Поділ на навчальну та тестову вибірки (80/20)
# stratify=y зберігає однакову частку дефолтів в обох вибірках
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nНавчальна вибірка: {X_train.shape[0]} записів")
print(f"Тестова вибірка: {X_test.shape[0]} записів")
print(f"Частка дефолтів у train: {y_train.mean():.2%}, у test: {y_test.mean():.2%}")


Навчальна вибірка: 26064 записів
Тестова вибірка: 6517 записів
Частка дефолтів у train: 21.82%, у test: 21.82%


In [19]:
# 3. Масштабування
# fit — лише на train, щоб інформація з тестової вибірки не «протікала» в модель
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [20]:
# 4. Навчання моделі логістичної регресії
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)
print("\nМодель навчено")


Модель навчено


In [21]:
# 5. Прогнози на тренувальній та тестовій вибірках
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

# Ймовірності дефолту
y_train_proba = model.predict_proba(X_train_scaled)[:, 1]
y_test_proba = model.predict_proba(X_test_scaled)[:, 1]

print(f"\nПрогнозована частка дефолтів у train: {y_train_pred.mean():.2%}")
print(f"Прогнозована частка дефолтів у test: {y_test_pred.mean():.2%}")

# Коефіцієнти моделі — які ознаки найбільше впливають на прогноз
coef_df = pd.DataFrame({'Ознака': X.columns, 'Коефіцієнт': model.coef_[0]}) \
            .sort_values('Коефіцієнт')
fig = px.bar(coef_df, x='Коефіцієнт', y='Ознака', orientation='h',
             color='Коефіцієнт', color_continuous_scale=['#264653', '#F4A261'],
             title='Коефіцієнти логістичної регресії')
fig.update_layout(height=600)
fig.show()


Прогнозована частка дефолтів у train: 14.94%
Прогнозована частка дефолтів у test: 15.07%



---

## Завдання 5: Оцінка якості моделі (4 бали)

**Що потрібно зробити:**
1. Побудуйте confusion matrix.
2. Обчисліть основні метрики (accuracy, precision, recall, f1).
3. Побудуйте ROC-криву та обчисліть AUC
4. Проаналізуйте важливість ознак.
5. Зробіть висновки про якість моделі та які ознаки найбільше впливають на прогноз.


In [22]:
# 1. Confusion matrix

from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, roc_curve, roc_auc_score,
                             classification_report)

labels = ['0 - сплачено', '1 - дефолт']

for name, y_true, y_pred in [('Train', y_train, y_train_pred), ('Test', y_test, y_test_pred)]:
    cm = confusion_matrix(y_true, y_pred)
    fig = px.imshow(cm, text_auto=True, x=labels, y=labels,
                    color_continuous_scale=['#FFFFFF', '#F4A261', '#264653'],
                    labels={'x': 'Прогноз', 'y': 'Факт', 'color': 'Кількість'},
                    title=f'Confusion matrix — {name}')
    fig.update_layout(height=450, width=550)
    fig.show()

tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()
print(f"Test: TN = {tn}, FP = {fp}, FN = {fn}, TP = {tp}")
print(f"Пропущено дефолтів (FN): {fn} з {fn + tp} ({fn / (fn + tp):.1%})")

Test: TN = 4848, FP = 247, FN = 687, TP = 735
Пропущено дефолтів (FN): 687 з 1422 (48.3%)


In [23]:
# 2. Основні метрики

metrics_df = pd.DataFrame({
    'Train': [accuracy_score(y_train, y_train_pred),
              precision_score(y_train, y_train_pred),
              recall_score(y_train, y_train_pred),
              f1_score(y_train, y_train_pred)],
    'Test': [accuracy_score(y_test, y_test_pred),
             precision_score(y_test, y_test_pred),
             recall_score(y_test, y_test_pred),
             f1_score(y_test, y_test_pred)]
}, index=['Accuracy', 'Precision', 'Recall', 'F1-score']).round(3)

display(metrics_df)

print("\nДетальний звіт (Test):")
print(classification_report(y_test, y_test_pred, target_names=labels))

,Train,Test
Accuracy,0.853,0.857
Precision,0.737,0.748
Recall,0.505,0.517
F1-score,0.600,0.611



Детальний звіт (Test):
              precision    recall  f1-score   support

0 - сплачено       0.88      0.95      0.91      5095
  1 - дефолт       0.75      0.52      0.61      1422

    accuracy                           0.86      6517
   macro avg       0.81      0.73      0.76      6517
weighted avg       0.85      0.86      0.85      6517



In [24]:
# 3. ROC-крива та AUC

fpr_train, tpr_train, _ = roc_curve(y_train, y_train_proba)
fpr_test, tpr_test, _ = roc_curve(y_test, y_test_proba)
auc_train = roc_auc_score(y_train, y_train_proba)
auc_test = roc_auc_score(y_test, y_test_proba)

fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr_train, y=tpr_train, mode='lines',
                         name=f'Train (AUC = {auc_train:.3f})', line=dict(color='#264653', width=2)))
fig.add_trace(go.Scatter(x=fpr_test, y=tpr_test, mode='lines',
                         name=f'Test (AUC = {auc_test:.3f})', line=dict(color='#F4A261', width=2)))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Випадкова модель',
                         line=dict(color='gray', dash='dash')))
fig.update_layout(title='ROC-крива', xaxis_title='False Positive Rate',
                  yaxis_title='True Positive Rate', height=550, width=700)
fig.show()

print(f"AUC Train: {auc_train:.3f}")
print(f"AUC Test:  {auc_test:.3f}")

AUC Train: 0.864
AUC Test:  0.866


In [25]:
# 4. Важливість ознак

importance_df = pd.DataFrame({
    'Ознака': X.columns,
    'Коефіцієнт': model.coef_[0],
    'Важливість': np.abs(model.coef_[0])
}).sort_values('Важливість', ascending=True)

importance_df['Вплив'] = np.where(importance_df['Коефіцієнт'] > 0,
                                  'Збільшує ризик дефолту', 'Зменшує ризик дефолту')

fig = px.bar(importance_df, x='Важливість', y='Ознака', orientation='h', color='Вплив',
             color_discrete_map={'Збільшує ризик дефолту': '#F4A261',
                                 'Зменшує ризик дефолту': '#264653'},
             title='Важливість ознак (|коефіцієнт| логістичної регресії)')
fig.update_layout(height=600)
fig.show()

display(importance_df.sort_values('Важливість', ascending=False)
        [['Ознака', 'Коефіцієнт']].round(3).reset_index(drop=True))

,Ознака,Коефіцієнт
0,loan_percent_income,1.222
1,loan_grade,1.116
2,loan_amnt,-0.449
3,person_home_ownership_OWN,-0.423
4,loan_intent_VENTURE,-0.394
5,person_home_ownership_RENT,0.383
6,loan_intent_EDUCATION,-0.318
7,loan_intent_PERSONAL,-0.204
8,person_income,-0.150
9,loan_int_rate,-0.067


**Висновки: оцінка якості моделі**

**Метрики**

**Перенавчання немає:** метрики на train і test майже однакові (accuracy 0,853 / 0,857, F1 0,600 / 0,611).

**Accuracy**= 0,857. Через дисбаланс класів ця метрика оманлива.
**Precision** = 0,748: 3 з 4 прогнозів дефолту правильні.

**Recall** = 0,517: модель знаходить лише половину реальних дефолтів (близько 735 з 1 422). Це головна слабкість моделі.

**F1-score** = 0,611: середня якість виявлення дефолтів.
Клас "сплачено" модель розпізнає добре (recall 0,95, F1 0,91), а клас "дефолт" значно гірше.

**AUC** = 0,866 (train 0,864): модель добре розрізняє класи. Випадковий позичальник із дефолтом отримає вищу прогнозовану ймовірність ризику, ніж випадковий позичальник, який сплатив кредит, приблизно у 87% випадків. Однакові значення на train і test також підтверджують, що перенавчання немає.

**Найважливіші ознаки**

**loan_percent_income (+1,22) і loan_grade (+1,12)** — головні фактори ризику дефолту.

**Оренда житла (RENT)** підвищує ризик, власне житло (OWN) знижує.

**Кредити на бізнес, освіту та особисті потреби** менш ризикові, ніж кредити на консолідацію боргів.

**Вищий дохід** знижує ризик.

**loan_int_rate і cb_person_default_on_file** майже не впливають, бо їхню інформацію вже містить loan_grade.

**Вік і стаж роботи** практично не впливають на прогноз.

**Загальний висновок**

Модель стабільна й точна в прогнозах дефолту (precision 0,75), але пропускає майже половину дефолтів (recall 0,52).
Для банку пропущений дефолт дорожчий за хибну відмову, тому модель варто покращити: додати class_weight='balanced' або знизити поріг класифікації до 0,3–0,4, щоб підвищити recall.


---

## Завдання 6: Оптимізація порогу та бізнес-аналіз (2 бали)

**Що потрібно зробити:**
1. Проаналізуйте метрики precision, recall, F1 при різних порогах класифікації (мінімум - 5 різних порогів). Візуалізуйте як змінюються метрики якості при зміні порогу.
2. Оберіть оптимальний поріг для бізнес-задачі - можна обрати виходячи з попереднього пункту, або додати своїх роздумів і обрати інший.



In [26]:
# 1. метрики якості при зміні порогу.

thresholds = np.arange(0.1, 0.91, 0.1).round(1)

rows = []
for t in thresholds:
    y_pred_t = (y_test_proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()
    rows.append({
        'Поріг': t,
        'Precision': precision_score(y_test, y_pred_t, zero_division=0),
        'Recall': recall_score(y_test, y_pred_t),
        'F1-score': f1_score(y_test, y_pred_t),
        'Accuracy': accuracy_score(y_test, y_pred_t),
        'FP (хибні відмови)': fp,
        'FN (пропущені дефолти)': fn
    })

threshold_df = pd.DataFrame(rows).round(3)
display(threshold_df)

,Поріг,Precision,Recall,F1-score,Accuracy,FP (хибні відмови),FN (пропущені дефолти)
0,0.1,0.375,0.893,0.529,0.652,2113,152
1,0.2,0.495,0.805,0.613,0.779,1165,278
2,0.3,0.596,0.710,0.648,0.832,686,412
3,0.4,0.682,0.613,0.646,0.853,407,550
4,0.5,0.748,0.517,0.611,0.857,247,687
5,0.6,0.805,0.414,0.546,0.850,142,834
6,0.7,0.875,0.314,0.462,0.840,64,976
7,0.8,0.920,0.203,0.333,0.822,25,1133
8,0.9,0.959,0.098,0.177,0.802,6,1283


In [27]:
# візуалізація

fig = go.Figure()
fig.add_trace(go.Scatter(x=threshold_df['Поріг'], y=threshold_df['Precision'],
                         mode='lines+markers', name='Precision', line=dict(color='#264653', width=3)))
fig.add_trace(go.Scatter(x=threshold_df['Поріг'], y=threshold_df['Recall'],
                         mode='lines+markers', name='Recall', line=dict(color='#F4A261', width=3)))
fig.add_trace(go.Scatter(x=threshold_df['Поріг'], y=threshold_df['F1-score'],
                         mode='lines+markers', name='F1-score', line=dict(color='#2A9D8F', width=3, dash='dash')))
fig.add_vline(x=0.5, line_dash='dot', line_color='gray', annotation_text='Поріг за замовчуванням')
fig.update_layout(title='Зміна метрик залежно від порогу класифікації',
                  xaxis_title='Поріг', yaxis_title='Значення метрики',
                  yaxis_range=[0, 1.05], height=500)
fig.show()

# Поріг з максимальним F1
best_f1 = threshold_df.loc[threshold_df['F1-score'].idxmax()]
print(f"Максимальний F1 = {best_f1['F1-score']:.3f} при порозі {best_f1['Поріг']}")

Максимальний F1 = 0.648 при порозі 0.3


In [28]:
# Припущення: пропущений дефолт (FN) коштує банку у 5 разів дорожче,
# ніж хибна відмова надійному клієнту (FP — втрачений прибуток від відсотків)
COST_FN = 5
COST_FP = 1

threshold_df['Вартість помилок'] = (threshold_df['FN (пропущені дефолти)'] * COST_FN +
                                    threshold_df['FP (хибні відмови)'] * COST_FP)
display(threshold_df[['Поріг', 'Precision', 'Recall', 'F1-score',
                      'FP (хибні відмови)', 'FN (пропущені дефолти)', 'Вартість помилок']])

best_cost = threshold_df.loc[threshold_df['Вартість помилок'].idxmin()]
print(f"Мінімальна вартість помилок при порозі {best_cost['Поріг']}")

fig = px.bar(threshold_df, x='Поріг', y='Вартість помилок',
             color_discrete_sequence=['#F4A261'],
             title=f'Сумарна вартість помилок (FN × {COST_FN} + FP × {COST_FP})')
fig.show()

,Поріг,Precision,Recall,F1-score,FP (хибні відмови),FN (пропущені дефолти),Вартість помилок
0,0.1,0.375,0.893,0.529,2113,152,2873
1,0.2,0.495,0.805,0.613,1165,278,2555
2,0.3,0.596,0.710,0.648,686,412,2746
3,0.4,0.682,0.613,0.646,407,550,3157
4,0.5,0.748,0.517,0.611,247,687,3682
5,0.6,0.805,0.414,0.546,142,834,4312
6,0.7,0.875,0.314,0.462,64,976,4944
7,0.8,0.920,0.203,0.333,25,1133,5690
8,0.9,0.959,0.098,0.177,6,1283,6421


Мінімальна вартість помилок при порозі 0.2


In [30]:
# Порівняння трьох варіантів порогу
candidates = {
    'Стандартний (0.5)': 0.5,
    f'Макс. F1 ({best_f1["Поріг"]})': best_f1['Поріг'],
    f'Мін. вартість ({best_cost["Поріг"]})': best_cost['Поріг']
}

rows = {}
for name, t in candidates.items():
    y_pred_t = (y_test_proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()
    rows[name] = {
        'Precision': precision_score(y_test, y_pred_t),
        'Recall': recall_score(y_test, y_pred_t),
        'F1-score': f1_score(y_test, y_pred_t),
        'Accuracy': accuracy_score(y_test, y_pred_t),
        'Виявлено дефолтів (TP)': tp,
        'Пропущено дефолтів (FN)': fn,
        'Хибні відмови (FP)': fp,
        'Відмовлено надійним клієнтам (%)': fp / (fp + tn) * 100,
        'Вартість помилок': fn * COST_FN + fp * COST_FP
    }

compare_df = pd.DataFrame(rows).round(3)
display(compare_df)

# Фінальний поріг — компроміс між виявленням дефолтів і втратою надійних клієнтів
OPTIMAL_THRESHOLD = 0.3
y_test_opt = (y_test_proba >= OPTIMAL_THRESHOLD).astype(int)

# Confusion matrix: стандартний поріг vs обраний
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Поріг 0.5', f'Поріг {OPTIMAL_THRESHOLD}'],
                    horizontal_spacing=0.15)

for i, preds in enumerate([y_test_pred, y_test_opt]):
    cm = confusion_matrix(y_test, preds)
    fig.add_trace(go.Heatmap(z=cm, x=labels, y=labels, text=cm, texttemplate='%{text}',
                             textfont=dict(size=16),
                             colorscale=[[0, '#FFFFFF'], [0.5, '#F4A261'], [1, '#264653']],
                             showscale=False),
                  row=1, col=i + 1)

fig.update_xaxes(title_text='Прогноз')
fig.update_yaxes(title_text='Факт', autorange='reversed')
fig.update_layout(title='Confusion matrix: стандартний vs обраний поріг', height=450, width=950)
fig.show()

# Підсумок змін
fn_05 = confusion_matrix(y_test, y_test_pred)[1, 0]
fn_opt = confusion_matrix(y_test, y_test_opt)[1, 0]
fp_05 = confusion_matrix(y_test, y_test_pred)[0, 1]
fp_opt = confusion_matrix(y_test, y_test_opt)[0, 1]
print(f"Обраний поріг: {OPTIMAL_THRESHOLD}")
print(f"Пропущених дефолтів: {fn_05} → {fn_opt} (на {fn_05 - fn_opt} менше)")
print(f"Хибних відмов: {fp_05} → {fp_opt} (на {fp_opt - fp_05} більше)")

,Стандартний (0.5),Макс. F1 (0.3),Мін. вартість (0.2)
Precision,0.748,0.596,0.495
Recall,0.517,0.710,0.805
F1-score,0.611,0.648,0.613
Accuracy,0.857,0.832,0.779
Виявлено дефолтів (TP),735.000,1010.000,1144.000
Пропущено дефолтів (FN),687.000,412.000,278.000
Хибні відмови (FP),247.000,686.000,1165.000
Відмовлено надійним клієнтам (%),4.848,13.464,22.866
Вартість помилок,3682.000,2746.000,2555.000


Обраний поріг: 0.3
Пропущених дефолтів: 687 → 412 (на 275 менше)
Хибних відмов: 247 → 686 (на 439 більше)


**Висновки**

Зі зниженням порогу recall зростає з 0,098 (поріг 0,9) до 0,893 (поріг 0,1), а precision падає з 0,959 до 0,375. Криві перетинаються приблизно на рівні 0,35.

Стандартний поріг 0,5 дає найвищу accuracy (0,857), але пропускає 687 дефолтів із 1 422, тобто майже половину.

**Максимальний F1** = 0,648 при порозі 0,3.

**Мінімальна вартість помилок (2 555) при порозі 0,2**, але ціною 1 165 хибних відмов: банк відмовив би близько 23% надійних клієнтів, і половина відмов була б помилковою (precision 0,495).

**Обраний поріг — 0,3**. Це компроміс:

recall зростає з 0,517 до 0,710, тобто виявляється 71% дефолтів;

пропущених дефолтів стає 412 замість 687 (на 275 менше);

хибних відмов 686 замість 247, тобто близько 13% надійних клієнтів;

вартість помилок (2 746) лише на 7% вища за мінімальну, але банк втрачає значно менше надійних клієнтів, ніж при порозі 0,2;

F1 при цьому максимальний (0,648).

Якщо банк орієнтується на мінімальний ризик і готовий втрачати клієнтів, можна обрати 0,2. Якщо важливіше зберегти клієнтську базу, підійде 0,3–0,4.


---

## Завдання 7 (Опціональне): Покращення моделі та висновки (2 бали)

**Що потрібно зробити:**

На цих же даних навчіть DecisionTreeClassifier та RandomForestClassifier. Яка з трьох моделей дає найкращий результат на тестовому наборі?

Поріг класифікації можна для порівняння якостей моделей використовувати стандартний 0.5, або той, що ви виявили, як оптимальний на попередньому кроці.


In [31]:
# 1. Навчання моделей

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Decision Tree — обмежуємо глибину, щоб уникнути перенавчання
dt_model = DecisionTreeClassifier(max_depth=8, min_samples_leaf=20, random_state=42)
dt_model.fit(X_train, y_train)

# Random Forest
rf_model = RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_leaf=5,
                                  random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Ймовірності дефолту для всіх трьох моделей
models_proba = {
    'Logistic Regression': {'train': y_train_proba, 'test': y_test_proba},
    'Decision Tree': {'train': dt_model.predict_proba(X_train)[:, 1],
                      'test': dt_model.predict_proba(X_test)[:, 1]},
    'Random Forest': {'train': rf_model.predict_proba(X_train)[:, 1],
                      'test': rf_model.predict_proba(X_test)[:, 1]}
}
print("Моделі навчено")

Моделі навчено


In [32]:
# 2. Порівняння метрик (пороги 0.5 та 0.3)

results = []
for threshold in [0.5, OPTIMAL_THRESHOLD]:
    for name, proba in models_proba.items():
        pred_test = (proba['test'] >= threshold).astype(int)
        pred_train = (proba['train'] >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, pred_test).ravel()
        results.append({
            'Поріг': threshold,
            'Модель': name,
            'Accuracy': accuracy_score(y_test, pred_test),
            'Precision': precision_score(y_test, pred_test),
            'Recall': recall_score(y_test, pred_test),
            'F1 (test)': f1_score(y_test, pred_test),
            'F1 (train)': f1_score(y_train, pred_train),
            'AUC': roc_auc_score(y_test, proba['test']),
            'FN': fn,
            'FP': fp
        })

results_df = pd.DataFrame(results).round(3)
display(results_df)

# Візуалізація метрик при обраному порозі
plot_df = results_df[results_df['Поріг'] == OPTIMAL_THRESHOLD] \
    .melt(id_vars='Модель', value_vars=['Precision', 'Recall', 'F1 (test)', 'AUC'],
          var_name='Метрика', value_name='Значення')

fig = px.bar(plot_df, x='Метрика', y='Значення', color='Модель', barmode='group',
             text='Значення',
             color_discrete_map={'Logistic Regression': '#264653',
                                 'Decision Tree': '#2A9D8F',
                                 'Random Forest': '#F4A261'},
             title=f'Порівняння моделей на тестовій вибірці (поріг {OPTIMAL_THRESHOLD})')
fig.update_traces(textposition='outside')
fig.update_layout(yaxis_range=[0, 1.1], height=500)
fig.show()

,Поріг,Модель,Accuracy,Precision,Recall,F1 (test),F1 (train),AUC,FN,FP
0,0.5,Logistic Regression,0.857,0.748,0.517,0.611,0.600,0.866,687,247
1,0.5,Decision Tree,0.923,0.942,0.690,0.797,0.808,0.912,441,60
2,0.5,Random Forest,0.932,0.973,0.707,0.819,0.843,0.930,416,28
3,0.3,Logistic Regression,0.832,0.596,0.710,0.648,0.639,0.866,412,686
4,0.3,Decision Tree,0.917,0.883,0.714,0.789,0.806,0.912,407,135
5,0.3,Random Forest,0.917,0.847,0.753,0.797,0.866,0.930,351,193


**Найкраща модель — Random Forest**. Вона лідирує за всіма ключовими метриками на тестовій вибірці.

**Порівняння з логістичною регресією**

Обидві деревні моделі значно перевершують логістичну регресію: AUC зростає з 0,866 до 0,912–0,930, F1 з 0,648 до 0,789–0,797.

Найбільша різниця в precision: 0,596 у регресії проти 0,85–0,88 у дерев. При тому самому recall дерева дають у 3–5 разів менше хибних відмов (135–193 проти 686).

**Перенавчання**

**Логістична регресія:** перенавчання немає (F1 train 0,639, test 0,648).

**Decision Tree**: різниця невелика (0,806 / 0,789), тож обмеження глибини спрацювало.

**Random Forest:** помірна різниця (0,866 / 0,797 при порозі 0,3).

Поріг 0,3 був оптимальним для логістичної регресії, але не для дерев. У Random Forest при порозі 0,5 F1 вищий (0,819), precision 0,973, і лише 28 хибних відмов.

**Бізнес-вартість помилок** (FN × 5 + FP × 1):

**RF** 0,3: 351 × 5 + 193 = 1 948 (найменша);

**RF** 0,5: 416 × 5 + 28 = 2 108;

**DT** 0,3: 2 170;

**LR** 0,3: 2 746.

Отже, з погляду бізнесу найкращий варіант — **Random Forest з порогом 0,3**: найменше пропущених дефолтів і найнижча сумарна вартість помилок.

Якщо банку важливіше не втрачати надійних клієнтів, підійде поріг 0,5 (лише 28 хибних відмов).

In [33]:
# 3. ROC-криві всіх моделей

colors = {'Logistic Regression': '#264653', 'Decision Tree': '#2A9D8F', 'Random Forest': '#F4A261'}

fig = go.Figure()
for name, proba in models_proba.items():
    fpr, tpr, _ = roc_curve(y_test, proba['test'])
    auc = roc_auc_score(y_test, proba['test'])
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name=f'{name} (AUC = {auc:.3f})',
                             line=dict(color=colors[name], width=3)))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Випадкова модель',
                         line=dict(color='gray', dash='dash')))
fig.update_layout(title='ROC-криві моделей (test)', xaxis_title='False Positive Rate',
                  yaxis_title='True Positive Rate', height=550, width=750)
fig.show()

На ROC-кривій лінії дерев лежать вище, особливо в зоні низького FPR: при мінімумі хибних тривог вони виявляють близько 70% дефолтів, а регресія лише близько 30%.

In [34]:
# 4. Важливість ознак у Random Forest

rf_importance = pd.DataFrame({
    'Ознака': X.columns,
    'Важливість': rf_model.feature_importances_
}).sort_values('Важливість', ascending=True)

fig = px.bar(rf_importance, x='Важливість', y='Ознака', orientation='h',
             color_discrete_sequence=['#F4A261'],
             title='Важливість ознак — Random Forest')
fig.update_layout(height=600)
fig.show()

**Висновки: порівняння моделей**

**Найкраща модель — Random Forest**. Вона лідирує за всіма ключовими метриками на тестовій вибірці.

**Важливість ознак (Random Forest)**

**Найважливіші:** loan_percent_income (0,27), loan_grade (0,16), person_income (0,14), person_home_ownership_RENT (0,11), loan_int_rate (0,09).

Результати узгоджуються з логістичною регресією: навантаження кредиту на дохід і рейтинг є головними факторами в обох моделях.

Відмінність: у Random Forest person_income і loan_int_rate важливіші, бо дерева вловлюють нелінійний вплив, якого регресія не бачила.

cb_person_default_on_file і person_home_ownership_OTHER майже не впливають.

**Загальний висновок**

Random Forest — найточніша модель (AUC 0,930, F1 0,797–0,819) з найменшою кількістю пропущених дефолтів.

Логістична регресія поступається в точності, але краще інтерпретується: її коефіцієнти показують напрямок впливу кожної ознаки. Це важливо, якщо банку потрібно пояснювати клієнтам причину відмови.

Рекомендація: Random Forest як основна модель для оцінки ризику, логістична регресія як допоміжна для пояснення рішень.